## Import and collect data descriptors
In this section, we will import the necessary libraries and collect data descriptors for our toxicity prediction project. Data descriptors are essential for understanding the structure and characteristics of our dataset, which will help us in the subsequent data cleaning and preprocessing steps.

In [1]:
# import nbimporter
# from descriptors import generate_all_features
%run descriptors.ipynb

All functions defined. Ready to process datasets.


In [2]:
from tdc.single_pred import Tox
from rdkit import Chem, RDLogger
from rdkit.Chem import Lipinski, Descriptors, Crippen,rdMolDescriptors, GraphDescriptors, QED
from rdkit.Chem import Fragments, AllChem
from rdkit.Chem.rdMolDescriptors import (
    CalcTPSA, CalcKappa1, CalcKappa2, CalcKappa3,
    CalcChi0v, CalcChi1v, CalcChi2v, CalcChi3v, CalcChi4v,
    CalcChi0n, CalcChi1n, CalcChi2n, CalcChi3n, CalcChi4n,
)

RDLogger.DisableLog('rdApp.*')

import os
from pathlib import Path
from copy import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

# Local utilities
import sys
sys.path.insert(0, '.')
import os

random_state = 0

MAIN_DIR = Path().resolve().parent.parent
PATH_DATA = MAIN_DIR / 'data'
PATH_PROCESSED_DATA = PATH_DATA / 'processed'

## 1. Import data from the OpenTox API
To import data from the [OpenTox API](https://opentox.org/api). We will then convert the JSON data into a pandas DataFrame for easier manipulation and analysis.

In [3]:
if not os.path.exists(PATH_DATA):
    os.makedirs(PATH_DATA)

data = Tox(name = 'LD50_Zhu', path= PATH_DATA / 'LD50_Zhu')
split = data.get_split()

Downloading...
100%|██████████| 707k/707k [00:00<00:00, 1.17MiB/s]
Loading...
Done!


In [4]:
for k in split.keys():
    print(f'{k} shape: {split[k].shape}')

train shape: (5170, 3)
valid shape: (738, 3)
test shape: (1477, 3)


In [5]:
split['train'].head()

,Drug_ID,Drug,Y
0,"Methane, tribromo-",BrC(Br)Br,2.343
1,Bromoethene (9CI),C=CBr,2.330
2,"1,1'-Biphenyl, hexabromo-",Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465
3,"Isothiocyanic acid, p-bromophenyl ester",S=C=Nc1ccc(Br)cc1,2.729
4,"Benzene, bromo-",Brc1ccccc1,1.765


## 2. Collect data smiles
Collect the SMILES (Simplified Molecular Input Line Entry System) representations of the chemical compounds in our dataset. SMILES is a widely used notation for describing the structure of chemical molecules, and it will be crucial for our toxicity prediction model.

In [6]:
split.keys()

dict_keys(['train', 'valid', 'test'])

In [7]:
# initialize a dictionary to store the dataframes with the new columns
df_dict = {}
for k in split.keys():
    df = split[k].copy()
    # convert the 'Drug' column to RDKit Mol objects
    df['mol'] = df['Drug'].apply(lambda x: Chem.MolFromSmiles(x))
    # convert the 'mol' canonical SMILES
    df['canonical_smiles'] = df['mol'].apply(lambda x: Chem.MolToSmiles(x, canonical=True) if x is not None else None)
    df_dict[k] = df

    del df

In [8]:
df_dict['train'].head()

,Drug_ID,Drug,Y,mol,canonical_smiles
0,"Methane, tribromo-",BrC(Br)Br,2.343,<rdkit.Chem.rdchem.Mol object at 0x000002327C3...,BrC(Br)Br
1,Bromoethene (9CI),C=CBr,2.330,<rdkit.Chem.rdchem.Mol object at 0x000002327D4...,C=CBr
2,"1,1'-Biphenyl, hexabromo-",Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465,<rdkit.Chem.rdchem.Mol object at 0x000002327D4...,Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br
3,"Isothiocyanic acid, p-bromophenyl ester",S=C=Nc1ccc(Br)cc1,2.729,<rdkit.Chem.rdchem.Mol object at 0x000002327D4...,S=C=Nc1ccc(Br)cc1
4,"Benzene, bromo-",Brc1ccccc1,1.765,<rdkit.Chem.rdchem.Mol object at 0x000002327D4...,Brc1ccccc1


## 3. Add Descriptors
In this step, we will add molecular descriptors to our dataset. Molecular descriptors are numerical values that describe the properties of chemical compounds, such as molecular weight, logP, and topological polar surface area. These descriptors will be used as features for our toxicity prediction model.

Descriptors selected for this project include:
- Morgan, Avalon, ErG fingerprints: from [Advancing ADMET Prediction: A Hybrid Machine Learning Framework (Partex ADMETrix)](https://github.com/partex-nv-opensource/tdc-submission/blob/main/src/ics-admet-prediction-tdc-versiontwo/report/Advancing%20ADMET%20Prediction_%20A%20Hybrid%20Machine%20Learning%20Framework%20(Partex%20ADMETrix).pdf)
- General Descriptors from [Advancing ADMET Prediction: A Hybrid Machine Learning Framework (Partex ADMETrix)](https://github.com/partex-nv-opensource/tdc-submission/blob/main/src/ics-admet-prediction-tdc-versiontwo/report/Advancing%20ADMET%20Prediction_%20A%20Hybrid%20Machine%20Learning%20Framework%20(Partex%20ADMETrix).pdf)
- Dragon descriptors from [Quantitative Structure−Activity Relationship Modeling of Rat Acute Toxicity by Oral Exposure](https://pubs.acs.org/doi/abs/10.1021/tx900189p?casa_token=vfBbuxuUCqEAAAAA:YAcI0r4Z3rtlRYP_l5H8OlTfdUh3DVlO6ws_h1NkhpaXH3-NrdI2-s5ghWWJbxfPQw-KhQIAwMi1Di3v)

In [9]:
for k in df_dict.keys():
    print(f'\n--- Processing {k} dataset ---')
    print(f'{k} initial shape: {df_dict[k].shape}')
    # function from descriptors.ipynb to generate all features and add them as columns to the dataframe
    df_dict[k] = generate_all_features(df_dict[k], mol_col='mol')
    print(f'{k} features shape: {df_dict[k].shape}')




--- Processing train dataset ---
train initial shape: (5170, 5)
Generating Fingerprints...
Computing RDKit Descriptors...
Computing Mordred Descriptors...


 47%|████▋     | 2424/5170 [00:31<01:02, 44.10it/s] 

c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 55%|█████▍    | 2831/5170 [00:45<01:47, 21.72it/s]

c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 96%|█████████▌| 1415/1477 [00:21<00:00, 66.27it/s]

c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 1477/1477 [00:22<00:00, 66.82it/s]


Feature generation complete. Total columns: 1821
test features shape: (1477, 1821)


## 4. Save the processed data
After adding the descriptors, we will save the processed DataFrame to a CSV file for later use in our machine learning pipeline. This will allow us to easily load the data in future steps without having to repeat the descriptor generation process.

In [10]:
df_dict['train'].columns

Index(['Drug_ID', 'Drug', 'Y', 'mol', 'canonical_smiles', 'morgan_fp',
       'avalon_fp', 'erg_fp', 'BalabanJ', 'BertzCT',
       ...
       'mordred_SRW10', 'mordred_TSRW10', 'mordred_MW', 'mordred_AMW',
       'mordred_WPath', 'mordred_WPol', 'mordred_Zagreb1', 'mordred_Zagreb2',
       'mordred_mZagreb1', 'mordred_mZagreb2'],
      dtype='object', length=1821)

In [11]:
# Export dataset (exclude non-serializable columns: mol)
cols_to_exclude = ['mol', 'morgan_fp', 'avalon_fp', 'erg_fp', 'fp']
export_cols = [c for c in df_dict['train'].columns if c not in cols_to_exclude]

# Create output directory if needed
os.makedirs(PATH_PROCESSED_DATA, exist_ok=True)

# Export to CSV (without no-serializable objects)
for key, df in df_dict.items():
    csv_path = f'{PATH_PROCESSED_DATA}/{key}.csv'
    df[export_cols].to_csv(csv_path, index=False)
    print(f"✓ {key} CSV exported: {csv_path} (shape: {df[export_cols].shape})")

# Also export as pickle for reproducibility (includes mol objects)
for key, df in df_dict.items():
    pkl_path = f'{PATH_PROCESSED_DATA}/{key}.pkl'
    df.to_pickle(pkl_path)
    print(f"✓ {key} dataset (with RDKit objects) exported: {pkl_path} (shape: {df.shape})")

✓ train CSV exported: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-D6310ZE-MLDD-25-26\toxicity-prediction\data\processed/train.csv (shape: (5170, 1817))
✓ valid CSV exported: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-D6310ZE-MLDD-25-26\toxicity-prediction\data\processed/valid.csv (shape: (738, 1817))
✓ test CSV exported: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-D6310ZE-MLDD-25-26\toxicity-prediction\data\processed/test.csv (shape: (1477, 1817))
✓ train dataset (with RDKit objects) exported: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-D6310ZE-MLDD-25-26\toxicity-prediction\data\processed/train.pkl (shape: (5170, 1821))
✓ valid dataset (with RDKit objects) exported: C:\Users\Tommaso\Documents\MEGAR2D2\Documenti\SUPSI\BACHELOR\3_anno_bech\primaverile\M-D6310ZE-MLDD-25-26\toxicity-prediction\data\processed/valid.pkl 